In [1]:
from pathlib import Path
import pandas as pd

In [2]:
YEAR = 2023
PROJECT_DIR = Path("../..").resolve()
print("Base Directory:", PROJECT_DIR)


Base Directory: /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline


In [3]:
from tennis_data_pipeline.validatation.tennis_data_uk.tournaments import (
    find_uk_inconsistent_tournaments, 
    find_uk_reused_tournament_ids, 
)

## Load Data

In [4]:
import re


def slugify(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def add_source_event_key(df):
    df = df.copy()

    # Date format drifts across seasons (e.g. "1/1/23" vs. "2023-01-01").
    df["Date"] = pd.to_datetime(df["Date"], format="mixed", errors="coerce")
    df["Year"] = df["Date"].dt.year

    df["source_event_key"] = (
        df["Year"].astype(str)
        + "_"
        + df["ATP"].astype(str)
        + "_"
        + df["Location"].map(slugify)
        + "_"
        + df["Tournament"].map(slugify)
    )

    return df


In [ ]:
def load_dirty_uk_atp_data(year: int) -> pd.DataFrame:
    df_uk = pd.read_csv(PROJECT_DIR / f"data/clean/tennis-data-uk/atp/atp_singles_results_{year}.csv")

    # Date format drifts across seasons (e.g. "1/1/23" vs. "2000-01-03").
    df_uk["Date"] = pd.to_datetime(df_uk["Date"], format="mixed", dayfirst=False)
    df_uk["Year"] = year

    # Nullable ints: ranks/points/set-scores are whole numbers but can be missing (e.g. retired matches).
    int_cols = [
        "ATP", "Year", "Best of",
        "WRank", "LRank", "WPts", "LPts",
        "W1", "L1", "W2", "L2", "W3", "L3", "W4", "L4", "W5", "L5",
        "Wsets", "Lsets",
    ]
    for col in int_cols:
        if col in df_uk.columns:
            df_uk[col] = pd.to_numeric(df_uk[col], errors="coerce").astype("Int64")

    # Everything left over is bookmaker odds; which bookmakers are present varies by year.
    known_cols = {
        "ATP", "Year", "Location", "Tournament", "Date", "Series", "Court",
        "Surface", "Round", "Best of", "Winner", "Loser", "Comment", *int_cols,
    }
    odds_cols = [col for col in df_uk.columns if col not in known_cols]
    df_uk[odds_cols] = df_uk[odds_cols].apply(pd.to_numeric, errors="coerce")

    for col in ["Series", "Court", "Surface", "Round", "Comment"]:
        if col in df_uk.columns:
            df_uk[col] = df_uk[col].astype("category")

    df_uk = add_source_event_key(df_uk)

    return df_uk


### Tennis Data UK

In [6]:
df_uk = load_dirty_uk_atp_data(YEAR)
print(list(df_uk.columns))

print(df_uk.head())


['ATP', 'Year', 'Location', 'Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank', 'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'PSW', 'PSL', 'MaxW', 'MaxL', 'AvgW', 'AvgL', 'source_event_key']
   ATP  Year  Location                Tournament       Date  Series    Court  \
0    1  2023  Adelaide  Adelaide International 1 2023-01-01  ATP250  Outdoor   
1    1  2023  Adelaide  Adelaide International 1 2023-01-01  ATP250  Outdoor   
2    1  2023  Adelaide  Adelaide International 1 2023-01-02  ATP250  Outdoor   
3    1  2023  Adelaide  Adelaide International 1 2023-01-02  ATP250  Outdoor   
4    1  2023  Adelaide  Adelaide International 1 2023-01-02  ATP250  Outdoor   

  Surface      Round  Best of  ...    Comment B365W  B365L   PSW   PSL  MaxW  \
0    Hard  1st Round        3  ...  Completed  1.91   1.91  1.93  1.95  1.99   
1    Hard  1st Round        3  ..

In [ ]:
display(list(df_uk.columns))
display(df_uk)


In [ ]:
{
    'ATP' : "uk_tourney_id",
    'Year' : "year",
    'Location' : "location",
    'Tournament' : "tournament",
    'Date' : "date",
    'Series' : "series",
    'Court' : "indoor_outdoor",
    'Surface' : "surface",
    'Round' : "round", 
    'Best of' : "best_of",
    'Winner' : "player_winner",
    'Loser' : "player_loser",
    'WRank' : "rank_winner",
    'LRank' : "rank_loser",
    'WPts' : "points_winner",
    'LPts' : "points_loser",
    'W1' : "set_1_score_winner",
    'L1' : "set_1_score_loser",
    'W2' : "set_2_score_winner",
    'L2' : "set_2_score_loser",
    'W3' : "set_3_score_winner",
    'L3' : "set_3_score_loser",
    'W4' : "set_4_score_winner",
    'L4' : "set_4_score_loser",
    'W5' : "set_5_score_winner",
    'L5' : "set_5_score_loser",
    'Wsets' : "sets_winner",
    'Lsets' : "sets_loser",
    'Comment' : "comment",
    'B365W' : "b365_odds_winner",
    'B365L' : "b365_odds_loser",
    'PSW' : "ps_odds_winner",
    'PSL' : "ps_odss_loser",
    'MaxW' : "max_odds_winner",
    'MaxL' : "max_odds_loser",
    'AvgW' : "avg_odds_winner",
    'AvgL' : "avg_odds_loser",
    'source_event_key'
}

In [11]:
KEY_COLUMNS = ["ATP", "Year", "Location"]
INFO_COLS =  ["Tournament", "Series", "Court", "Surface", "Best of"]

Get Tournaments and their information

In [12]:
from tennis_data_pipeline.validatation.tennis_data_uk.tournaments import (
    find_uk_inconsistent_tournaments, 
    find_uk_reused_tournament_ids, 
)

In [13]:
inconsistent_tourneys = find_uk_inconsistent_tournaments(df_uk, key_columns=KEY_COLUMNS, info_cols=INFO_COLS)
inconsistent_tourneys

All tournament attributes are consistent.


(Empty DataFrame
 Columns: [Tournament, Series, Court, Surface, Best of]
 Index: [],
 Empty DataFrame
 Columns: [ATP, Year, Location, Tournament, Date, Series, Court, Surface, Round, Best of, Winner, Loser, WRank, LRank, WPts, LPts, W1, L1, W2, L2, W3, L3, W4, L4, W5, L5, Wsets, Lsets, Comment, B365W, B365L, PSW, PSL, MaxW, MaxL, AvgW, AvgL, source_event_key]
 Index: []
 
 [0 rows x 38 columns])

In [13]:
reused_tourney_id = find_uk_reused_tournament_ids(df=df_uk, id_col= "ATP", disambiguating_cols=["Location", "Tournament"])
reused_tourney_id

(     Location  Tournament
 ATP                      
 58          2           2,
     ATP  Year   Location                       Tournament      Date  Series  \
 0    58  2023  Stockholm                      Nordic Open  10/16/23  ATP250   
 1    58  2023  Stockholm                      Nordic Open  10/16/23  ATP250   
 2    58  2023  Stockholm                      Nordic Open  10/16/23  ATP250   
 3    58  2023  Stockholm                      Nordic Open  10/16/23  ATP250   
 4    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 5    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 6    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 7    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 8    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 9    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 10   58  2023  Stockholm             

In [ ]:
tourney_df = (
    df_uk.groupby("source_event_key", as_index=False)
    .agg(
        atp_tournament_id=("ATP", "first"),
        year=("Year", "first"),
        tournament_name=("Tournament", "first"),
        location=("Location", "first"),
        surface=("Surface", "first"),
        court=("Court", "first"),
        **{"best_of_sets": ("Best of", "first")},
        start_date=("Date", "min"),
        end_date=("Date", "max"),
    )
    .sort_values(by="start_date")
)
tourney_df


,source_event_key,ATP,Year,Location,Tournament,Court,Surface,Best of,start_date,end_date
10,2023_1_adelaide_adelaide_international_1,1,2023,Adelaide,Adelaide International 1,Outdoor,Hard,3,2023-01-01,2023-01-08
21,2023_2_pune_maharashtra_open,2,2023,Pune,Maharashtra Open,Outdoor,Hard,3,2023-01-02,2023-01-07
43,2023_4_auckland_asb_classic,4,2023,Auckland,ASB Classic,Outdoor,Hard,3,2023-01-08,2023-01-14
32,2023_3_adelaide_adelaide_international_2,3,2023,Adelaide,Adelaide International 2,Outdoor,Hard,3,2023-01-09,2023-01-14
55,2023_5_melbourne_australian_open,5,2023,Melbourne,Australian Open,Outdoor,Hard,5,2023-01-16,2023-01-29
...,...,...,...,...,...,...,...,...,...,...
56,2023_60_vienna_vienna_open,60,2023,Vienna,Vienna Open,Indoor,Hard,3,2023-10-23,2023-10-29
57,2023_61_paris_bnp_paribas_masters,61,2023,Paris,BNP Paribas Masters,Indoor,Hard,3,2023-10-30,2023-11-05
58,2023_62_metz_open_de_moselle,62,2023,Metz,Open de Moselle,Indoor,Hard,3,2023-11-05,2023-11-11
59,2023_63_sofia_sofia_open,63,2023,Sofia,Sofia Open,Indoor,Hard,3,2023-11-07,2023-11-11


In [ ]:
display(
    df_uk.loc[
        df_uk["ATP"].isin([38]),
        ["ATP", "Location", "Tournament", "Series", "Court", "Surface", "Best of", "Date"],
    ].drop_duplicates()
)


In [ ]:
uk_tourneys = df_uk['Tournament'].unique()
print(uk_tourneys)

In [ ]:
df_timl.loc[~df_timl['tourney_name'].str.contains("Davis Cup"), "tourney_name"].unique()

In [ ]:
source_df = df_uk.copy()
canonical_df = df_timl.loc[~df_timl['tourney_name'].str.contains("Davis Cup")].copy()


source_tournaments = (
    source_df[["Tournament"]]
    .drop_duplicates()
    .rename(columns={"Tournament": "source_tournament_name"})
)

canonical_tournaments = (
    canonical_df[["tourney_id", "tourney_name"]]
    .drop_duplicates()
    .rename(
        columns={
            "tourney_id": "canonical_tournament_id",
            "tourney_name": "canonical_tournament_name",
        }
    )
)

tournament_crosswalk = source_tournaments.merge(
    canonical_tournaments,
    left_on="source_tournament_name",
    right_on="canonical_tournament_name",
    how="left",
    validate="one_to_one",
)

tournament_crosswalk["match_method"] = "exact_name"
tournament_crosswalk["confidence"] = 1.0
tournament_crosswalk["review_flag"] = (
    tournament_crosswalk["canonical_tournament_id"].isna()
)

In [ ]:
tournament_crosswalk

In [ ]:
tournament_crosswalk["year"] = 2023
tournament_crosswalk["source"] = "tennis_data_uk"

In [ ]:
print(tournament_crosswalk)